![](../img/330-banner.png)

# Tutorial 3

UBC 2025-26

## Outline

During this tutorial, we will focus on preprocessing - the necessary steps to perform to make the data meaningful for a learning algorithm.

All questions can be discussed with your classmates and the TAs - this is not a graded exercise!

In [31]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML

sys.path.append("../code/.")
from plotting_functions import *
from utils import *

pd.set_option("display.max_colwidth", 200)

from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

## `ColumnTransformer` on the California housing dataset 

In this notebook, you will practice features preprocessing using the [California housing dataset](https://www.kaggle.com/datasets/camnugent/california-housing-prices).

Let's start by loading the dataset (this is done for you):

In [32]:
housing_df = pd.read_csv("../data/housing.csv")
train_df, test_df = train_test_split(housing_df, test_size=0.1, random_state=123)

train_df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
6051,-117.75,34.04,22.0,2948.0,636.0,2600.0,602.0,3.1250,113600.0,INLAND
20113,-119.57,37.94,17.0,346.0,130.0,51.0,20.0,3.4861,137500.0,INLAND
14289,-117.13,32.74,46.0,3355.0,768.0,1457.0,708.0,2.6604,170100.0,NEAR OCEAN
13665,-117.31,34.02,18.0,1634.0,274.0,899.0,285.0,5.2139,129300.0,INLAND
14471,-117.23,32.88,18.0,5566.0,1465.0,6303.0,1458.0,1.8580,205000.0,NEAR OCEAN


Let's also add some new features that may help us with the prediction:

In [33]:
train_df = train_df.assign(
    rooms_per_household=train_df["total_rooms"] / train_df["households"]
)
test_df = test_df.assign(
    rooms_per_household=test_df["total_rooms"] / test_df["households"]
)

train_df = train_df.assign(
    bedrooms_per_household=train_df["total_bedrooms"] / train_df["households"]
)
test_df = test_df.assign(
    bedrooms_per_household=test_df["total_bedrooms"] / test_df["households"]
)

train_df = train_df.assign(
    population_per_household=train_df["population"] / train_df["households"]
)
test_df = test_df.assign(
    population_per_household=test_df["population"] / test_df["households"]
)

Finally, we are separating for you the target from the features:

In [34]:
# Let's keep both numeric and categorical columns in the data.
X_train = train_df.drop(columns=["median_house_value"])
y_train = train_df["median_house_value"]

X_test = test_df.drop(columns=["median_house_value"])
y_test = test_df["median_house_value"]

## Step 0: EDA
Let's get a sense for our dataset using the strategies we learned about last time. From this information, what kinds of preprocessing steps might we need to take here?

In [35]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18576 entries, 6051 to 19966
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   longitude                 18576 non-null  float64
 1   latitude                  18576 non-null  float64
 2   housing_median_age        18576 non-null  float64
 3   total_rooms               18576 non-null  float64
 4   total_bedrooms            18391 non-null  float64
 5   population                18576 non-null  float64
 6   households                18576 non-null  float64
 7   median_income             18576 non-null  float64
 8   median_house_value        18576 non-null  float64
 9   ocean_proximity           18576 non-null  object 
 10  rooms_per_household       18576 non-null  float64
 11  bedrooms_per_household    18391 non-null  float64
 12  population_per_household  18576 non-null  float64
dtypes: float64(12), object(1)
memory usage: 2.0+ MB


In [36]:
housing_df["ocean_proximity"].unique()

array(['NEAR BAY', '<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'ISLAND'],
      dtype=object)

In [37]:
train_df.describe()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,rooms_per_household,bedrooms_per_household,population_per_household
count,18576.000000,18576.000000,18576.000000,18576.000000,18391.000000,18576.000000,18576.000000,18576.000000,18576.000000,18576.000000,18391.000000,18576.000000
mean,-119.565888,35.627966,28.622255,2635.749677,538.229786,1428.578165,500.061100,3.862552,206292.067991,5.426067,1.097516,3.052349
std,1.999622,2.134658,12.588307,2181.789934,421.805266,1141.664801,383.044313,1.892491,115083.856175,2.512319,0.486266,10.020873
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000,0.846154,0.333333,0.692308
25%,-121.790000,33.930000,18.000000,1449.000000,296.000000,788.000000,280.000000,2.560225,119400.000000,4.439360,1.005888,2.430323
50%,-118.490000,34.250000,29.000000,2127.000000,435.000000,1167.000000,410.000000,3.527500,179300.000000,5.226415,1.048860,2.818868
75%,-118.010000,37.710000,37.000000,3145.000000,647.000000,1727.000000,606.000000,4.736900,263600.000000,6.051620,1.099723,3.283921
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000,141.909091,34.066667,1243.333333


## Step 1

Your turn now! Start by importing ColumnTranformer and make_column_transformer

In [38]:
from sklearn.compose import ColumnTransformer, make_column_transformer

## Step 2

Next, group features by type (numerical or categorical). You may also want to save the target separately. 

In [39]:
numeric_features = [
    "longitude",
    "latitude",
    "housing_median_age",
    "rooms_per_household",
    "bedrooms_per_household",
    "population_per_household",
    "households",
    "median_income"
    ]

categorical_features = ["ocean_proximity"]

target = "median_house_value"

### <font color='aqua'> we don't need to add total rooms and total bedrooms ==> because??

## Step 3

Create a ColumnTransformer for your features. The transformer should include imputation and scaling for numeric features, and encoding for categorical features (which type of encoding?)

In [40]:
numeric_transformer = make_pipeline(
    SimpleImputer(strategy="median"), StandardScaler()
)
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = make_column_transformer(
    (numeric_transformer, numeric_features),
    (categorical_transformer, categorical_features)
)

## Step 4

Visualize the transformed training set as a dataframe

In [41]:
X_train_pp = preprocessor.fit_transform(X_train)
column_names = numeric_features + list(
    preprocessor.named_transformers_["onehotencoder"].get_feature_names_out(categorical_features)
)

pd.DataFrame(X_train_pp, columns=column_names)



,longitude,latitude,housing_median_age,rooms_per_household,bedrooms_per_household,population_per_household,households,median_income,ocean_proximity_<1H OCEAN,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,0.908140,-0.743917,-0.526078,-0.210591,-0.083813,0.126398,0.266135,-0.389736,0.0,1.0,0.0,0.0,0.0
1,-0.002057,1.083123,-0.923283,4.726412,11.166631,-0.050132,-1.253312,-0.198924,0.0,1.0,0.0,0.0,0.0
2,1.218207,-1.352930,1.380504,-0.273606,-0.025391,-0.099240,0.542873,-0.635239,0.0,0.0,0.0,0.0,1.0
3,1.128188,-0.753286,-0.843842,0.122307,-0.280310,0.010183,-0.561467,0.714077,0.0,1.0,0.0,0.0,0.0
4,1.168196,-1.287344,-0.843842,-0.640266,-0.190617,0.126808,2.500924,-1.059242,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18571,0.733102,-0.804818,0.586095,0.063110,-0.099558,0.071541,-0.966131,-0.118182,1.0,0.0,0.0,0.0,0.0
18572,1.163195,-1.057793,-1.161606,0.235096,-0.163397,0.007458,0.728235,0.357500,1.0,0.0,0.0,0.0,0.0
18573,-1.097293,0.797355,-1.876574,0.211892,-0.135305,0.044029,0.514155,0.934269,1.0,0.0,0.0,0.0,0.0
18574,-1.437367,1.008167,1.221622,-0.273382,-0.149822,-0.132875,-0.454427,0.006578,0.0,0.0,0.0,1.0,0.0


In [42]:
#show the column names after preprocessing


## Step 5

Finally, let's train a classifier (or even better, for practice, a baseline and another regressor): 
- create a pipeline with the preprocessor and a regressor of your choice.
- use the pipeline to perform cross-validation

In [43]:
from sklearn.svm import SVR

svr_pipeline = make_pipeline(preprocessor, SVR())
scores = cross_validate(svr_pipeline, X_train, y_train, cv=10, return_train_score=True)
pd.DataFrame(pd.DataFrame(scores).mean())

,0
fit_time,13.356808
score_time,2.066904
test_score,-0.048004
train_score,-0.047404


In [44]:
knn_pipeline = make_pipeline(preprocessor, KNeighborsRegressor())
scores = cross_validate(knn_pipeline, X_train, y_train, cv=10, return_train_score=True)
pd.DataFrame(pd.DataFrame(scores).mean())

,0
fit_time,0.032811
score_time,0.092977
test_score,0.700108
train_score,0.804845


## <font color='red'>Recap/comprehension questions</font>

- Do we have to preprocess the target column too?
- If we only plan to use a Decision Tree as classifier, do we still need to scale the numerical features?
### <font color='aqua'> - No because it is splitting by feature value and each feature is independently handled but still good practice to do if you want to switch models</font>

- If the dataset included an ordinal feature "Neighbourhood desirability", with numerical labels 1 (poor), 2 (good) and 3 (excellent), would we need to apply an ordinal encoder to it?
### <font color='aqua'> - No, because it's already numerical. but the values of 1, 2 and 3 are lables and don't mean anything</font>

- Why do we add the argument `drop="if_binary"` to `OneHotEncoder` when dealing with categorical features with only two possible values? What would be the disadvantages of not doing so?
### <font color='aqua'> - To save space, the two columns created are going to say the same thing but opposite. it creates unnecessary correlation between features in linear models</font>
